# 02 — Stationarity Analysis

**Goal:** Run ADF tests on each series, determine differencing order,
and populate the `adf_pvalue` fields in `preprocessing_config.yaml`.

**Process:**
1. Load raw series
2. For each: test level → if non-stationary, log-transform and test → difference and test
3. Record ADF p-value and differencing order
4. Write results back to preprocessing_config.yaml

**Output:** Updated `preprocessing_config.yaml` with `adf_pvalue` populated.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'backend'))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / 'backend' / '.env')  # explicit path

import numpy as np
import pandas as pd
import yaml
from statsmodels.tsa.stattools import adfuller

from data.ingestion import load_series_config


In [2]:
from data.ingestion import fetch_all_series


raw = fetch_all_series()
config = load_series_config()
config_by_id = {s['id']: s for s in config}

  ✓ UTEDUH — 253 observations
  ✓ SMU49000006562100001SA — 253 observations
  ✓ UTNA — 253 observations
  ✓ UTURN — 253 observations
  ✓ LBSSA49 — 253 observations
  ✓ UTICLAIMS — 1110 observations
  ✓ UTCCLAIMS — 1109 observations
  ✓ SMU49000000500000003 — 229 observations
  ✓ CES6562000001 — 255 observations
  ✓ JTS6200JOL — 254 observations
  ✓ JTS6200JOR — 254 observations
  ✓ JTS6200QUR — 254 observations
  ✓ JTS6200HIL — 254 observations
  ✓ CES6500000003 — 241 observations
  ✓ ECIALLCIV — 84 observations
  ✓ UNRATE — 255 observations
  ✓ CPIAUCSL — 255 observations
  ✓ FEDFUNDS — 255 observations


In [3]:
def run_adf(series, name=''):
    """Run ADF test, return (statistic, p-value, is_stationary)."""
    clean = series.dropna()
    if len(clean) < 20:
        return None, None, False
    result = adfuller(clean, autolag='AIC')
    stat, pval = result[0], result[1]
    stationary = pval < 0.05
    print(f'  {name:40s} ADF stat={stat:8.3f}  p={pval:.4f}  {"✓ stationary" if stationary else "✗ non-stationary"}')
    return stat, pval, stationary

In [4]:
results = []

for sid, series in raw.items():
    sc = config_by_id.get(sid)
    if sc is None:
        continue
    
    print(f'\n--- {sid} ---')
    
    # Level test
    _, pval_level, is_stat = run_adf(series, f'{sid} (level)')
    
    if is_stat:
        results.append({'id': sid, 'differencing': 0, 'log_transform': False, 'adf_pvalue': round(pval_level, 4)})
        continue
    
    # Log + level test (if config says log_transform)
    if sc.get('log_transform', False):
        log_series = np.log(series[series > 0])
        _, pval_log, is_stat_log = run_adf(log_series, f'{sid} (log level)')
        if is_stat_log:
            results.append({'id': sid, 'differencing': 0, 'log_transform': True, 'adf_pvalue': round(pval_log, 4)})
            continue
        # Log + first difference
        diff1 = log_series.diff().dropna()
    else:
        diff1 = series.diff().dropna()
    
    _, pval_d1, is_stat_d1 = run_adf(diff1, f'{sid} (diff=1)')
    
    if is_stat_d1:
        results.append({'id': sid, 'differencing': 1, 'log_transform': sc.get('log_transform', False), 'adf_pvalue': round(pval_d1, 4)})
    else:
        # Second difference (rare)
        diff2 = diff1.diff().dropna()
        _, pval_d2, _ = run_adf(diff2, f'{sid} (diff=2)')
        results.append({'id': sid, 'differencing': 2, 'log_transform': sc.get('log_transform', False), 'adf_pvalue': round(pval_d2, 4) if pval_d2 else None})

results_df = pd.DataFrame(results)
results_df


--- UTEDUH ---
  UTEDUH (level)                           ADF stat=   1.230  p=0.9962  ✗ non-stationary
  UTEDUH (log level)                       ADF stat=  -0.946  p=0.7725  ✗ non-stationary
  UTEDUH (diff=1)                          ADF stat= -13.968  p=0.0000  ✓ stationary

--- SMU49000006562100001SA ---
  SMU49000006562100001SA (level)           ADF stat=   0.928  p=0.9934  ✗ non-stationary
  SMU49000006562100001SA (log level)       ADF stat=  -1.147  p=0.6958  ✗ non-stationary
  SMU49000006562100001SA (diff=1)          ADF stat= -14.428  p=0.0000  ✓ stationary

--- UTNA ---
  UTNA (level)                             ADF stat=   0.274  p=0.9761  ✗ non-stationary
  UTNA (log level)                         ADF stat=  -0.150  p=0.9442  ✗ non-stationary
  UTNA (diff=1)                            ADF stat= -16.636  p=0.0000  ✓ stationary

--- UTURN ---
  UTURN (level)                            ADF stat=  -2.591  p=0.0949  ✗ non-stationary
  UTURN (diff=1)                           AD

,id,differencing,log_transform,adf_pvalue
0,UTEDUH,1,True,0.0000
1,SMU49000006562100001SA,1,True,0.0000
2,UTNA,1,True,0.0000
3,UTURN,1,False,0.0000
4,LBSSA49,1,False,0.0000
5,UTICLAIMS,0,False,0.0000
6,UTCCLAIMS,0,False,0.0022
7,SMU49000000500000003,1,False,0.0000
8,CES6562000001,1,True,0.0000
9,JTS6200JOL,1,False,0.0007


In [5]:
# TODO: Write adf_pvalue results back into preprocessing_config.yaml
# This should be done manually after reviewing the results above.
# Open backend/features/preprocessing_config.yaml and fill in adf_pvalue fields.

print('\n=== Copy these into preprocessing_config.yaml ===')
for _, row in results_df.iterrows():
    print(f"  {row['id']}: adf_pvalue={row['adf_pvalue']}, differencing={row['differencing']}")


=== Copy these into preprocessing_config.yaml ===
  UTEDUH: adf_pvalue=0.0, differencing=1
  SMU49000006562100001SA: adf_pvalue=0.0, differencing=1
  UTNA: adf_pvalue=0.0, differencing=1
  UTURN: adf_pvalue=0.0, differencing=1
  LBSSA49: adf_pvalue=0.0, differencing=1
  UTICLAIMS: adf_pvalue=0.0, differencing=0
  UTCCLAIMS: adf_pvalue=0.0022, differencing=0
  SMU49000000500000003: adf_pvalue=0.0, differencing=1
  CES6562000001: adf_pvalue=0.0, differencing=1
  JTS6200JOL: adf_pvalue=0.0007, differencing=1
  JTS6200JOR: adf_pvalue=0.0013, differencing=1
  JTS6200QUR: adf_pvalue=0.0, differencing=1
  JTS6200HIL: adf_pvalue=0.0, differencing=1
  CES6500000003: adf_pvalue=0.0, differencing=2
  ECIALLCIV: adf_pvalue=0.0, differencing=2
  UNRATE: adf_pvalue=0.0, differencing=1
  CPIAUCSL: adf_pvalue=0.0, differencing=2
  FEDFUNDS: adf_pvalue=0.0047, differencing=1
